# Python and C++ extension

## Importing library

In [ ]:
import numpy as np
import argparse
import os.path
import scipy.sparse
import vtk
from pypolydim import polydim, gedim
from pypolydim.export_vtk_utilities import ExportVTKUtilities
from pypolydim.assembler_utilities import assembler_utilities
import matplotlib
from typing import List

import sys
sys.path.insert(1, '../')
import other_utilities as other_ut

In [ ]:
geometry_utilities_config = gedim.GeometryUtilitiesConfig()
geometry_utilities_config.tolerance1_d = 1.0e-6
geometry_utilities_config.tolerance2_d = 1.0e-12
geometry_utilities = gedim.GeometryUtilities(geometry_utilities_config)
mesh_utilities = gedim.MeshUtilities()
vtk_utilities = ExportVTKUtilities()

### Initialize

In [ ]:
# Export folder
export_file_path = "./Export/Test_1"
if not os.path.exists(export_file_path):
    os.makedirs(export_file_path)

# Mesh file path
export_mesh_path = export_file_path + "/Mesh"
if not os.path.exists(export_mesh_path):
    os.makedirs(export_mesh_path)

# Solution file path
export_solution_path = export_file_path + "/Solution"
if not os.path.exists(export_solution_path):
    os.makedirs(export_solution_path)

## Non-Linear Equation

Solving the following equation on square $\bar{\Omega} = [0, 1] \times [0, 1]$

$$
\begin{cases}
- \nabla \cdot (\nabla u) + u \nabla \cdot u = g & \text{in } \Omega\\
u = 0.0 & \text{in } ∂Ω
\end{cases}
$$

where $u = 16 xy(1-x)(1-y)$.

The weak form of the problem becomes, find $u \in V := H^1_0(\Omega)$
$$
\int_{\Omega} \nabla u \nabla v + \int_{\Omega} u \nabla \cdot u v - \int_{\Omega} g v = 0 \quad \forall v \in V \Leftrightarrow f(u; v) := f_1(u; v) + f_2(u; v) + f_3(u; v) = 0 \quad \forall v \in V
$$

Using Newton schema, we solve for each $k$ iteration the problem
$$
J_f [\partial u]_{|_{u_k}} = - f(u_k; v) = 0 \quad \forall v \in V
$$
where $J_f [\partial u]_{|_{u_k}}$ is the evaluation of the derivative (Jacobian) of $J_f$ in the point $u_k$ along the unknown direction of $\partial u$.

After computations, we find the linear system, on each $k$ iteration, fixed $u_k$ find $\partial u$ s.t.

$$
\int_{\Omega} \nabla \partial u \cdot \nabla v + \int_{\Omega} \nabla \cdot u_k \partial u \ v + \int_{\Omega} u_k \nabla \cdot \partial u \ v = - \int_{\Omega} \nabla u_k \cdot \nabla v - \int_{\Omega} \nabla u_k \cdot u_k \ v + \int_{\Omega} g v 
$$

### Define Simulation Parameters

Set geometry parameters

In [ ]:
pde_domain = polydim.pde_tools.mesh.pde_mesh_utilities.PDE_Domain_2D()
pde_domain.vertices = np.array([[0.0, 1.0, 1.0, 0.0],
                                [0.0, 0.0, 1.0, 1.0],
                                [0.0, 0.0, 0.0, 0.0]])
pde_domain.shape_type = polydim.pde_tools.mesh.pde_mesh_utilities.PDE_Domain_2D.Domain_Shape_Types.parallelogram
pde_domain.area = 1.0

In [ ]:
mesh_type = polydim.pde_tools.mesh.pde_mesh_utilities.MeshGenerator_Types_2D.triangular
method_type = polydim.pde_tools.local_space_pcc_2_d.MethodTypes.fem_pcc
method_order = 1
mesh_size = 0.01

In [ ]:
mesh_data = gedim.MeshMatrices()
mesh = gedim.MeshMatricesDAO(mesh_data)

polydim.pde_tools.mesh.pde_mesh_utilities.create_mesh_2_d(geometry_utilities,
                                                          mesh_utilities,
                                                          mesh_type,
                                                          pde_domain,
                                                          mesh_size,
                                                          mesh)
mesh_geometric_data = polydim.pde_tools.mesh.pde_mesh_utilities.compute_mesh_2_d_geometry_data(geometry_utilities, mesh_utilities, mesh)

In [ ]:
vtk_utilities.export_mesh(export_mesh_path, mesh)
other_ut.plot_mesh(mesh)

In [ ]:
info_internal = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.none)
info_internal.marker = 0

info_dirichlet = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.strong)
info_dirichlet.marker = 1

boundary_info = {
    0: info_internal,
    1: info_dirichlet,
    2: info_dirichlet,
    3: info_dirichlet,
    4: info_dirichlet,
    5: info_dirichlet,
    6: info_dirichlet,
    7: info_dirichlet,
    8: info_dirichlet
}

In [ ]:
mesh_connectivity_data = polydim.pde_tools.mesh.MeshMatricesDAO_mesh_connectivity_data(mesh)

trial_reference_element_data = polydim.pde_tools.local_space_pcc_2_d.create_reference_element(method_type, method_order)
test_reference_element_data = polydim.pde_tools.local_space_pcc_2_d.create_reference_element(method_type, method_order)

dof_manager = polydim.pde_tools.do_fs.DOFsManager()

trial_mesh_dofs_info = polydim.pde_tools.local_space_pcc_2_d.set_mesh_do_fs_info(trial_reference_element_data, mesh, boundary_info)
trial_dofs_data = dof_manager.create_do_fs_2_d(trial_mesh_dofs_info, mesh_connectivity_data)
test_mesh_dofs_info = polydim.pde_tools.local_space_pcc_2_d.set_mesh_do_fs_info(trial_reference_element_data, mesh, boundary_info)
test_dofs_data = dof_manager.create_do_fs_2_d(test_mesh_dofs_info, mesh_connectivity_data)

## Run Newton Algorithm

Set Newton parameters

In [ ]:
residual_norm = 1.0
solution_norm = 1.0;
newton_tol = 1.0e-6
max_iterations = 7
num_iteration = 1

Set problem data

In [ ]:
def exact_du_function(x, y, z):
    return 0.0

def exact_solution_function(x, y, z):
    return 16.0 * x * y * (1.0 - x) * (1.0 - y) + 1.1

def exact_gradient_solution_function(x, y, z):
    return np.array([\
        16.0 * (1.0 - 2.0 * x) * y * (1.0 - y),\
        16.0 * (1.0 - 2.0 * y) * x * (1.0 - x),\
        0.0])

def exact_laplacian_solution_function(x, y, z):
    return -32.0 * (x * (1.0 - x) + y * (1.0 - y))

In [ ]:
def strong_solution_function(marker, x, y, z):  
    return exact_solution_function(x, y, z)

u_strong = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_strong_solution(geometry_utilities,
                                                                                  mesh,
                                                                                  mesh_geometric_data,
                                                                                  trial_mesh_dofs_info,
                                                                                  trial_dofs_data,
                                                                                  trial_reference_element_data,
                                                                                  strong_solution_function)

In [ ]:
def source_term_function(x, y, z):
    u_lap = exact_laplacian_solution_function(x, y, z)
    u_grad = exact_gradient_solution_function(x, y, z)
    u = exact_solution_function(x, y, z)

    return -u_lap + u * (u_grad[0] + u_grad[1])

f_g = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_source_term(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       test_dofs_data,
                                                                       trial_reference_element_data,
                                                                       test_reference_element_data,
                                                                       source_term_function)                                                                       

In [ ]:
def diffusion_term_function(x, y, z, u_k, u_grad_k):  
    return 1.0
def advection_term_function(x, y, z, u_k, u_grad_k):  
    return np.array([u_k, u_k, 0.0])
def reaction_term_function(x, y, z, u_k, u_grad_k):  
    return (u_grad_k[0] + u_grad_k[1])
def adv_non_linear_function(x : float, y : float, z : float, u_k : float, u_grad_k : List[float]) -> List[float]:  
    return [u_grad_k[0], u_grad_k[1], 0.0]
def rct_non_linear_function(x, y, z, u_k, u_grad_k):  
    return u_k * (u_grad_k[0] + u_grad_k[1])

Set Initial Solution

In [ ]:
n_dofs = trial_dofs_data.number_do_fs
n_strongs = trial_dofs_data.number_strongs

u_k = np.zeros(n_dofs)
du_strong = np.zeros(n_strongs)

Run Newton algorithm

In [ ]:
while num_iteration < max_iterations and residual_norm > newton_tol * solution_norm: 
    elliptic_operator = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_elliptic_operator(geometry_utilities,
                                                                                                 mesh,
                                                                                                 mesh_geometric_data,
                                                                                                 trial_dofs_data,
                                                                                                 test_dofs_data,
                                                                                                 trial_reference_element_data,
                                                                                                 test_reference_element_data,
                                                                                                 u_k,
                                                                                                 u_strong,
                                                                                                 diffusion_term_function,
                                                                                                 advection_term_function,
                                                                                                 reaction_term_function)
    A = other_ut.make_np_sparse(elliptic_operator.operator_dofs)
    A_D = other_ut.make_np_sparse(elliptic_operator.operator_strong)

    f_rct = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_source_term(geometry_utilities,
                                                                               mesh,
                                                                               mesh_geometric_data,
                                                                               trial_dofs_data,
                                                                               test_dofs_data,
                                                                               trial_reference_element_data,
                                                                               test_reference_element_data,
                                                                               u_k,
                                                                               u_strong,
                                                                               rct_non_linear_function)
    f_adv = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_source_term_gradients(geometry_utilities,
                                                                                         mesh,
                                                                                         mesh_geometric_data,
                                                                                         trial_dofs_data,
                                                                                         test_dofs_data,
                                                                                         trial_reference_element_data,
                                                                                         test_reference_element_data,
                                                                                         u_k,
                                                                                         u_strong,
                                                                                         adv_non_linear_function)

    rhs = f_g - f_adv - f_rct
    du = scipy.sparse.linalg.spsolve(A, rhs)
    
    u_k = u_k + du
    
    du_error_L2 = polydim.pde_tools.assembler_utilities.pcc_2_d.compute_error_l2(geometry_utilities,
                                                                                 mesh,
                                                                                 mesh_geometric_data,
                                                                                 trial_dofs_data,
                                                                                 trial_reference_element_data,
                                                                                 du,
                                                                                 du_strong,
                                                                                 exact_du_function)
    u_error_L2 = polydim.pde_tools.assembler_utilities.pcc_2_d.compute_error_l2(geometry_utilities,
                                                                                mesh,
                                                                                mesh_geometric_data,
                                                                                trial_dofs_data,
                                                                                trial_reference_element_data,
                                                                                u_k,
                                                                                u_strong,
                                                                                exact_solution_function)
    u_error_H1 = polydim.pde_tools.assembler_utilities.pcc_2_d.compute_error_h1(geometry_utilities,
                                                                                mesh,
                                                                                mesh_geometric_data,
                                                                                trial_dofs_data,
                                                                                trial_reference_element_data,
                                                                                u_k,
                                                                                u_strong,
                                                                                exact_gradient_solution_function)
       
    solution_norm = u_error_L2.numeric_norm_l2;
    residual_norm = du_error_L2.numeric_norm_l2;

    print("dofs", "errorL2", "errorH1", "residual", "iteration", "max_iteration")
    print(trial_dofs_data.number_do_fs, '{:.2e}'.format(u_error_L2.error_l2 / u_error_L2.numeric_norm_l2), '{:.2e}'.format(u_error_H1.error_h1 / u_error_H1.numeric_norm_h1), '{:.16e}'.format(residual_norm / solution_norm), '{:d}'.format(num_iteration), '{:d}'.format(max_iterations))
    
    num_iteration = num_iteration + 1

### Plot Solution

In [ ]:
u_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                                          trial_dofs_data,
                                                                                          u_k,
                                                                                          u_strong)

In [ ]:
u_exact_on_dofs = polydim.pde_tools.assembler_utilities.pcc_2_d.evaluate_function_on_dofs(geometry_utilities,
                                                                                mesh,
                                                                                mesh_geometric_data,
                                                                                trial_dofs_data,
                                                                                trial_reference_element_data,
                                                                        exact_solution_function)
u_exact_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                                          trial_dofs_data,
                                                                                          u_exact_on_dofs.function_dofs,
                                                                                          u_exact_on_dofs.function_strong)


In [ ]:
vtk_utilities.export_solution_2(export_solution_path + '/Solution',
                                mesh, 
                                u_on_cell0Ds.numeric_solution)
other_ut.plot_solution(mesh, u_on_cell0Ds.numeric_solution, "numeric_solution") 
other_ut.plot_solution(mesh, u_exact_on_cell0Ds.numeric_solution, "exact_solution") 

In [ ]:
u_on_quadrature = polydim.pde_tools.assembler_utilities.pcc_2_d.evaluate_solution_on_quadrature_points(geometry_utilities,
                                                                                mesh,
                                                                                mesh_geometric_data,
                                                                                trial_dofs_data,
                                                                                trial_reference_element_data,
                                                                                u_k,
                                                                                u_strong,
                                                                                     exact_solution_function,
                                                                                exact_gradient_solution_function)

In [ ]:
vtk_utilities.export_points(export_solution_path + '/solution_on_quadrature.vtu',
                            u_on_quadrature.quadrature_points,
                            { "num_sol": u_on_quadrature.numeric_solution })